In [8]:
import tensorflow as tf
import keras_tuner as kt
import numpy as np
EPOCHS = 10
IMG_WIDTH=30
IMG_HEIGHT=30
num_of_categories=43


In [2]:
# load the preprocessed data
train, val, test = np.load("train_processed.npz"), np.load("val_processed.npz"), np.load("test_processed.npz")
X_train, y_train, X_val, y_val, X_test, y_test = train["X"], train["y"], val["X"], val["y"], test["X"], test["y"]

In [9]:
def build_model(hp):
    regularizer_strength = hp.Float("l2", min_value=1e-4, max_value=1e-1, sampling="log") 
    dropout_rate = hp.Float("dropout", min_value=0.0, max_value=0.5, step=0.1)
    model = tf.keras.models.Sequential([
        tf.keras.layers.Conv2D(
            32, (3, 3), activation="relu", padding="same",
            kernel_initializer='he_normal', 
            kernel_regularizer=tf.keras.regularizers.l2(regularizer_strength), 
            input_shape=(IMG_WIDTH, IMG_HEIGHT, 3)
        ),
        tf.keras.layers.BatchNormalization(),

        tf.keras.layers.Conv2D(
            64, (3, 3), activation="relu", padding="same", 
            kernel_initializer='he_normal', 
            kernel_regularizer=tf.keras.regularizers.l2(regularizer_strength)
        ),
        tf.keras.layers.MaxPooling2D(pool_size=(2, 2)),
        tf.keras.layers.BatchNormalization(),

        tf.keras.layers.Flatten(),
        
        tf.keras.layers.Dense(
            128, activation="relu", 
            kernel_initializer='he_normal', 
            kernel_regularizer=tf.keras.regularizers.l2(regularizer_strength)
        ),
        tf.keras.layers.Dropout(dropout_rate),
        tf.keras.layers.BatchNormalization(),
        
        tf.keras.layers.Dense(num_of_categories, activation="softmax")
    ])
    model.compile(
        optimizer="nadam",  
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )
    print("Model constructed.")
    return model

In [4]:
# one hot encoding
y_train = tf.keras.utils.to_categorical(y_train)
y_val = tf.keras.utils.to_categorical(y_val)
y_test = tf.keras.utils.to_categorical(y_test)

In [ ]:
tuner = kt.RandomSearch(
    hypermodel=build_model,
    objective="val_accuracy",
    max_trials=10,
    executions_per_trial=1,
    directory="cnn_tuning",
    project_name="gtsrb"
)

tuner.search(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=32,
    verbose=1
)

best_hps = tuner.get_best_hyperparameters(1)[0]
print("Best hyperparameters:")
print("L2:", best_hps.get("l2"))
print("Dropout:", best_hps.get("dropout"))

best_model = tuner.get_best_models(1)[0]
best_model.evaluate(X_test, y_test)

# Evaluate neural network performance on validation set
best_model.evaluate(X_val, y_val, verbose=2, batch_size=32)

# Evaluate neural network performance on test set
best_model.evaluate(X_test, y_test, verbose=2, batch_size=32)

Trial 10 Complete [00h 12m 54s]
val_accuracy: 0.9424724578857422

Best val_accuracy So Far: 0.993880033493042
Total elapsed time: 02h 16m 33s
Best hyperparameters:
L2: 0.0001349128716661596
Dropout: 0.2
Best hyperparameters:
L2: 0.0001349128716661596
Dropout: 0.2
Model constructed.
c:\Users\smile\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\saving\saving_lib.py:802: UserWarning: Skipping variable loading for optimizer 'nadam', because it has 2 variables whereas the saved optimizer has 31 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))
395/395 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step - accuracy: 0.9540 - loss: 0.3883
154/154 - 1s - 9ms/step - accuracy: 0.9539 - loss: 0.3862
395/395 - 3s - 8ms/step - accuracy: 0.9537 - loss: 0.3846


The accuracy of the last epoch on train and the accuracy on validation are close, meaning there is little overfitting. 

The accuracy on test set is 92.79%, a 143.93% improvement from the baseline model accuracy.

In [15]:
# save the model to a file
filename = "model.keras"
best_model.save(filename)
print(f"Model saved to {filename}.")

Model saved to model.keras.
